# Spider Silkom External Material-Property Evaluation

This notebook evaluates whether the current structure-based SMD mechanical-property predictor can explain spider silk **material-level** mechanical properties.

Important interpretation: each row in `compressed_species_silk_db.csv` is taxon/species-level material performance for a mixed silk, while the predictor scores individual protein structures. Therefore the analysis aggregates multiple predicted silk-protein structures per `taxon_key` before comparing with experimental material `toughness` and `tensile_strength`.

In [ ]:
from __future__ import annotations
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import os
import time
import traceback
import json
import math

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_ROOT = Path('/mnt/nas/jianquanzhao/data/mprl/spider_silkom_database')
PDB_DIR = DATA_ROOT / 'colab-pdb'
OUT_ROOT = Path('output/evaluate-mechanical-predictor/evalutaed-spider_silkom')
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

STRUCTURE_CACHE = TABLE_DIR / 'spider_silkom_structure_predictions.csv'
MAX_WORKERS = min(16, os.cpu_count() or 4)

print('DATA_ROOT =', DATA_ROOT)
print('OUT_ROOT =', OUT_ROOT)
print('MAX_WORKERS =', MAX_WORKERS)

## 1. Parse Taxon-Level Material Labels and Protein Structures

`compressed_species_silk_db.csv` provides taxon-level mixed-silk material labels. `silk_sequence_performance_db.csv` provides the silk-protein FASTA records. ColabFold PDB files are matched from each FASTA basename by appending `_relaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb`.

In [ ]:
compressed = pd.read_csv(DATA_ROOT / 'infor_csv' / 'compressed_species_silk_db.csv')
seq = pd.read_csv(DATA_ROOT / 'infor_csv' / 'silk_sequence_performance_db.csv')
compressed = compressed.rename(columns={c: c.strip() for c in compressed.columns})
seq = seq.rename(columns={c: c.strip() for c in seq.columns})

label_cols = [
    'taxon_key', 'silk_types_included', 'toughness', 'tensile_strength',
    'family', 'genus', 'species', 'fragment_count', 'total_combined_len',
]
labels = compressed[label_cols].copy()
labels['has_material_label'] = labels[['toughness', 'tensile_strength']].notna().all(axis=1)

def structure_id_from_fasta(fasta_file):
    if pd.isna(fasta_file):
        return None
    return Path(str(fasta_file)).name.replace('.fasta', '')

def pdb_path_from_fasta(fasta_file):
    structure_id = structure_id_from_fasta(fasta_file)
    if structure_id is None:
        return None
    return PDB_DIR / f'{structure_id}_relaxed_rank_001_alphafold2_ptm_model_1_seed_000.pdb'

seq['structure_id'] = seq['fasta_file'].map(structure_id_from_fasta)
seq['pdb_path'] = seq['fasta_file'].map(pdb_path_from_fasta)
seq['pdb_exists'] = seq['pdb_path'].map(lambda path: bool(path and Path(path).exists()))

pair_table = (
    seq.merge(labels[['taxon_key', 'has_material_label', 'silk_types_included']], on='taxon_key', how='inner')
    .dropna(subset=['structure_id'])
)
pair_table['pdb_exists'] = pair_table['pdb_path'].map(lambda path: bool(path and Path(path).exists()))
unique_pairs = pair_table[['taxon_key', 'silk_type', 'fasta_file', 'structure_id', 'pdb_path', 'sequence']].drop_duplicates()

print('compressed taxon rows:', len(compressed))
print('taxa with material labels:', int(labels['has_material_label'].sum()))
print('sequence rows:', len(seq))
print('unique taxon/silk/structure pairs:', len(unique_pairs))
print('pairs with relaxed PDB:', int(unique_pairs['pdb_path'].map(lambda path: bool(path and Path(path).exists())).sum()))
unique_pairs.head()

## 2. Evaluate Every Unique Protein Structure

The structure-level predictions are cached in `spider_silkom_structure_predictions.csv`. If the cache already exists, this cell skips completed structures and only evaluates missing ones.

In [ ]:
def build_structure_work_table(pair_table: pd.DataFrame) -> pd.DataFrame:
    work = pair_table[pair_table['pdb_exists']].copy()
    keep_cols = ['structure_id', 'taxon_key', 'silk_type', 'fasta_file', 'pdb_path']
    return work[keep_cols].drop_duplicates(subset=['structure_id']).reset_index(drop=True)

work = build_structure_work_table(pair_table)
if STRUCTURE_CACHE.exists() and STRUCTURE_CACHE.stat().st_size > 0:
    cache = pd.read_csv(STRUCTURE_CACHE)
else:
    cache = pd.DataFrame()

done_ids = set(cache.loc[cache.get('ok', False).fillna(False), 'structure_id'].astype(str)) if not cache.empty and 'structure_id' in cache.columns else set()
pending = work[~work['structure_id'].astype(str).isin(done_ids)].copy()
print(f'unique structures with PDB: {len(work)}')
print(f'cached ok structures: {len(done_ids)}')
print(f'pending structures: {len(pending)}')

_CALC = None

def _init_worker():
    global _CALC
    from model.reward_module.terminal_reward.calculator import HbondTopologyTerminalRewardCalculator
    _CALC = HbondTopologyTerminalRewardCalculator()
    for estimator in getattr(_CALC.artifact.model, 'estimators_', []):
        model = getattr(estimator, 'estimator', estimator)
        if hasattr(model, 'n_jobs'):
            model.n_jobs = 1

def _evaluate_structure(row):
    global _CALC
    if _CALC is None:
        _init_worker()
    result = {
        'structure_id': row['structure_id'],
        'taxon_key': row['taxon_key'],
        'silk_type': row['silk_type'],
        'fasta_file': row['fasta_file'],
        'pdb_path': str(row['pdb_path']),
        'ok': False,
    }
    try:
        score = _CALC.evaluate_pdb(row['pdb_path'])
        result.update({
            'ok': True,
            'predicted_toughness_v127': score.toughness,
            'predicted_strength_v128': score.strength,
            'terminal_reward': score.reward,
            'normalized_toughness': score.normalized_toughness,
            'normalized_strength': score.normalized_strength,
            'mean_plddt': score.structure_quality * 100.0 if score.structure_quality is not None else np.nan,
            **score.features,
            'hbond_count': score.feature_diagnostics.get('hbond_count', np.nan),
            'hydrogen_atom_count': score.feature_diagnostics.get('hydrogen_atom_count', np.nan),
            'warnings': '; '.join(score.feature_diagnostics.get('warnings', [])),
            'error': '',
        })
    except Exception as exc:
        result.update({'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc(limit=4)})
    return result

rows = [] if cache.empty else cache.to_dict('records')
if len(pending):
    start = time.time()
    records = pending.to_dict('records')
    with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=_init_worker) as executor:
        futures = [executor.submit(_evaluate_structure, row) for row in records]
        for i, future in enumerate(as_completed(futures), start=1):
            rows.append(future.result())
            if i % 200 == 0:
                pd.DataFrame(rows).drop_duplicates(subset=['structure_id'], keep='last').to_csv(STRUCTURE_CACHE, index=False)
                elapsed = time.time() - start
                print(f'{i}/{len(records)} pending complete; {i / elapsed:.2f} structures/s')
    pd.DataFrame(rows).drop_duplicates(subset=['structure_id'], keep='last').to_csv(STRUCTURE_CACHE, index=False)

structure_predictions = pd.read_csv(STRUCTURE_CACHE)
print('structure prediction rows:', len(structure_predictions))
print('ok rows:', int(structure_predictions['ok'].fillna(False).sum()))
structure_predictions.head()

## 3. Aggregate Structure Predictions to Taxon-Level Material Predictions

Two aggregation schemes are reported:

- **structure-weighted:** every unique predicted structure contributes equally.
- **type-balanced:** structures are averaged within each `taxon_key + silk_type`, then silk types are averaged equally. This is the primary interpretation because material properties are determined by a mixture of silk types rather than by how many fragments happened to be present in the database.

In [ ]:
pred_cols = [
    'predicted_toughness_v127', 'predicted_strength_v128', 'terminal_reward',
    'normalized_toughness', 'normalized_strength', 'mean_plddt', 'sequence_length',
    'hbond_per_residue', 'seq_class_nonlocal_per_residue', 'strong_nonlocal_fraction',
    'strong_nonlocal_per_residue', 'nonlocal_backbone_backbone_per_residue',
    'hbond_contact_order', 'hbond_count', 'hydrogen_atom_count',
]

seq_unique = seq[['taxon_key', 'silk_type', 'fasta_file', 'structure_id', 'sequence']].dropna(subset=['structure_id']).drop_duplicates(subset=['taxon_key', 'silk_type', 'structure_id'])
seq_pred = seq_unique.merge(
    structure_predictions.drop(columns=[c for c in ['taxon_key', 'silk_type', 'fasta_file'] if c in structure_predictions.columns]),
    on='structure_id',
    how='left',
)
seq_pred['ok'] = seq_pred['ok'].fillna(False).astype(bool)
seq_pred.to_csv(TABLE_DIR / 'spider_silkom_sequence_structure_prediction_pairs.csv', index=False)
ok = seq_pred[seq_pred['ok']].copy()

structure_weighted = ok.groupby('taxon_key').agg(
    n_structures=('structure_id', 'nunique'),
    n_silk_types=('silk_type', 'nunique'),
    silk_types_observed=('silk_type', lambda x: ','.join(sorted(set(map(str, x))))),
    **{f'{col}_mean': (col, 'mean') for col in pred_cols},
    **{f'{col}_std': (col, 'std') for col in pred_cols},
).reset_index()
structure_weighted = labels.merge(structure_weighted, on='taxon_key', how='left')
structure_weighted.to_csv(TABLE_DIR / 'spider_silkom_taxon_aggregated_predictions.csv', index=False)

type_level = ok.groupby(['taxon_key', 'silk_type']).agg(
    n_type_structures=('structure_id', 'nunique'),
    **{f'{col}_type_mean': (col, 'mean') for col in pred_cols},
).reset_index()
type_level.to_csv(TABLE_DIR / 'spider_silkom_taxon_silk_type_predictions.csv', index=False)

type_balanced = type_level.groupby('taxon_key').agg(
    n_silk_types_type_balanced=('silk_type', 'nunique'),
    n_structures_type_balanced=('n_type_structures', 'sum'),
    silk_types_type_balanced=('silk_type', lambda x: ','.join(sorted(set(map(str, x))))),
    **{f'{col}_type_balanced_mean': (f'{col}_type_mean', 'mean') for col in pred_cols},
    **{f'{col}_type_balanced_std_across_types': (f'{col}_type_mean', 'std') for col in pred_cols},
).reset_index()
type_balanced = labels.merge(type_balanced, on='taxon_key', how='left')
type_balanced.to_csv(TABLE_DIR / 'spider_silkom_taxon_type_balanced_predictions.csv', index=False)

print('taxa with any structure prediction:', int(structure_weighted['n_structures'].notna().sum()))
print('taxa with label and prediction:', int((type_balanced['has_material_label'] & type_balanced['n_structures_type_balanced'].notna()).sum()))
type_balanced.head()

## 4. Correlation and Top-k Screening Analysis

In [ ]:
def safe_corr(frame, x_col, y_col):
    sub = frame[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) < 3 or sub[x_col].nunique() < 2 or sub[y_col].nunique() < 2:
        return {'n': len(sub), 'pearson': np.nan, 'pearson_p': np.nan, 'spearman': np.nan, 'spearman_p': np.nan}
    pearson = stats.pearsonr(sub[x_col], sub[y_col])
    spearman = stats.spearmanr(sub[x_col], sub[y_col])
    return {'n': int(len(sub)), 'pearson': float(pearson.statistic), 'pearson_p': float(pearson.pvalue), 'spearman': float(spearman.statistic), 'spearman_p': float(spearman.pvalue)}

metric_rows = []
for agg_name, frame, suffix in [
    ('structure_weighted', structure_weighted, '_mean'),
    ('type_balanced', type_balanced, '_type_balanced_mean'),
]:
    pairs = [
        ('experimental_toughness_vs_predicted_toughness', 'toughness', f'predicted_toughness_v127{suffix}'),
        ('experimental_toughness_vs_predicted_strength', 'toughness', f'predicted_strength_v128{suffix}'),
        ('experimental_toughness_vs_terminal_reward', 'toughness', f'terminal_reward{suffix}'),
        ('experimental_strength_vs_predicted_strength', 'tensile_strength', f'predicted_strength_v128{suffix}'),
        ('experimental_strength_vs_predicted_toughness', 'tensile_strength', f'predicted_toughness_v127{suffix}'),
        ('experimental_strength_vs_terminal_reward', 'tensile_strength', f'terminal_reward{suffix}'),
        ('experimental_strength_vs_sequence_length', 'tensile_strength', f'sequence_length{suffix}'),
        ('experimental_toughness_vs_sequence_length', 'toughness', f'sequence_length{suffix}'),
    ]
    for comparison, exp_col, pred_col in pairs:
        row = {'aggregation': agg_name, 'comparison': comparison, 'experimental_col': exp_col, 'predicted_col': pred_col}
        row.update(safe_corr(frame, pred_col, exp_col))
        metric_rows.append(row)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABLE_DIR / 'spider_silkom_correlation_metrics.csv', index=False)
metrics

In [ ]:
def topk_rows(frame, aggregation, exp_col, score_col, fractions=(0.1, 0.2, 0.3)):
    sub = frame[['taxon_key', exp_col, score_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    n = len(sub)
    rows = []
    if n == 0:
        return rows
    true_order = sub.sort_values(exp_col, ascending=False)
    pred_order = sub.sort_values(score_col, ascending=False)
    for frac in fractions:
        k = max(1, int(math.ceil(n * frac)))
        true_top = true_order.head(k)
        pred_top = pred_order.head(k)
        true_ids = set(true_top['taxon_key'].astype(str))
        pred_ids = set(pred_top['taxon_key'].astype(str))
        hits = sorted(true_ids & pred_ids)
        hit_rate = len(hits) / k
        baseline_rate = k / n
        rows.append({
            'aggregation': aggregation,
            'experimental_target': exp_col,
            'score': score_col,
            'top_fraction': frac,
            'n': n,
            'k': k,
            'hit_count': len(hits),
            'hit_rate': hit_rate,
            'baseline_rate': baseline_rate,
            'enrichment_factor': hit_rate / baseline_rate if baseline_rate else np.nan,
            'mean_experimental_in_pred_topk': float(pred_top[exp_col].mean()),
            'mean_experimental_overall': float(sub[exp_col].mean()),
            'topk_lift_vs_overall': float(pred_top[exp_col].mean() / sub[exp_col].mean()) if sub[exp_col].mean() else np.nan,
            'hit_taxon_keys': ','.join(hits),
            'pred_top_taxon_keys': ','.join(pred_top['taxon_key'].astype(str).tolist()),
            'true_top_taxon_keys': ','.join(true_top['taxon_key'].astype(str).tolist()),
        })
    return rows

top_rows = []
for aggregation, frame, suffix in [
    ('structure_weighted', structure_weighted, '_mean'),
    ('type_balanced', type_balanced, '_type_balanced_mean'),
]:
    top_rows += topk_rows(frame, aggregation, 'toughness', f'predicted_toughness_v127{suffix}')
    top_rows += topk_rows(frame, aggregation, 'tensile_strength', f'predicted_strength_v128{suffix}')
    top_rows += topk_rows(frame, aggregation, 'toughness', f'terminal_reward{suffix}')
    top_rows += topk_rows(frame, aggregation, 'tensile_strength', f'terminal_reward{suffix}')

topk = pd.DataFrame(top_rows)
topk.to_csv(TABLE_DIR / 'spider_silkom_topk_analysis.csv', index=False)
topk[(topk['aggregation'] == 'type_balanced') & (topk['top_fraction'] == 0.1)]

## 5. Figures

In [ ]:
plot_frame = type_balanced[type_balanced['has_material_label'] & type_balanced['predicted_toughness_v127_type_balanced_mean'].notna()].copy()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
scatter_specs = [
    ('predicted_toughness_v127_type_balanced_mean', 'toughness', 'Predicted SMD toughness vs material toughness'),
    ('predicted_strength_v128_type_balanced_mean', 'tensile_strength', 'Predicted SMD strength vs material tensile strength'),
    ('predicted_strength_v128_type_balanced_mean', 'toughness', 'Predicted SMD strength vs material toughness'),
    ('predicted_toughness_v127_type_balanced_mean', 'tensile_strength', 'Predicted SMD toughness vs material tensile strength'),
]
for ax, (x_col, y_col, title) in zip(axes.ravel(), scatter_specs):
    sub = plot_frame[[x_col, y_col, 'n_silk_types_type_balanced']].dropna()
    sc = ax.scatter(sub[x_col], sub[y_col], c=sub['n_silk_types_type_balanced'], cmap='viridis', s=30, alpha=0.8, edgecolor='none')
    corr = safe_corr(plot_frame, x_col, y_col)
    ax.set_title(f"{title}\nPearson={corr['pearson']:.2f}, Spearman={corr['spearman']:.2f}, n={corr['n']}")
    ax.set_xlabel(x_col.replace('_type_balanced_mean', ''))
    ax.set_ylabel(y_col)
fig.colorbar(sc, ax=axes.ravel().tolist(), label='number of silk types')
fig.savefig(FIG_DIR / 'spider_silkom_predicted_vs_material_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

corr_cols = [
    'toughness', 'tensile_strength',
    'predicted_toughness_v127_type_balanced_mean', 'predicted_strength_v128_type_balanced_mean', 'terminal_reward_type_balanced_mean',
    'sequence_length_type_balanced_mean', 'hbond_per_residue_type_balanced_mean', 'strong_nonlocal_per_residue_type_balanced_mean', 'hbond_contact_order_type_balanced_mean',
]
corr_df = plot_frame[corr_cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap='coolwarm')
labels_for_plot = [c.replace('_type_balanced_mean', '').replace('predicted_', 'pred_') for c in corr_cols]
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(labels_for_plot, rotation=45, ha='right')
ax.set_yticklabels(labels_for_plot)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr_df.values[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Spearman correlation')
ax.set_title('Spider silkom taxon-level Spearman correlations')
fig.tight_layout()
fig.savefig(FIG_DIR / 'spider_silkom_spearman_heatmap.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, target, title in zip(axes, ['toughness', 'tensile_strength'], ['Top material toughness enrichment', 'Top material tensile strength enrichment']):
    sub = topk[(topk['aggregation'] == 'type_balanced') & (topk['experimental_target'] == target)].copy()
    for score in sub['score'].unique():
        ss = sub[sub['score'] == score].sort_values('top_fraction')
        label = score.replace('_type_balanced_mean', '').replace('predicted_', 'pred_')
        ax.plot((ss['top_fraction'] * 100).astype(int), ss['enrichment_factor'], marker='o', label=label)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Top fraction (%)')
    ax.set_ylabel('Enrichment factor')
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'spider_silkom_topk_enrichment.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(plot_frame['toughness'].dropna(), bins=25, color='#4C78A8', alpha=0.8)
axes[0].set_title('Material toughness distribution')
axes[0].set_xlabel('toughness')
axes[0].set_ylabel('taxon count')
axes[1].hist(plot_frame['tensile_strength'].dropna(), bins=25, color='#F58518', alpha=0.8)
axes[1].set_title('Material tensile strength distribution')
axes[1].set_xlabel('tensile_strength')
fig.tight_layout()
fig.savefig(FIG_DIR / 'spider_silkom_material_property_distribution.png', dpi=200)
plt.show()

## 6. Summary and Conclusion

In [ ]:
summary = {
    'compressed_taxon_rows': int(len(compressed)),
    'taxa_with_material_labels': int(labels['has_material_label'].sum()),
    'sequence_rows': int(len(seq)),
    'unique_sequence_structure_pairs': int(len(unique_pairs)),
    'unique_predicted_structures_ok': int(structure_predictions['ok'].fillna(False).sum()),
    'taxa_with_any_structure_prediction': int(structure_weighted['n_structures'].notna().sum()),
    'taxa_with_label_and_prediction': int(plot_frame.shape[0]),
    'primary_correlation_metrics': metrics[
        (metrics['aggregation'] == 'type_balanced')
        & metrics['comparison'].isin([
            'experimental_toughness_vs_predicted_toughness',
            'experimental_strength_vs_predicted_strength',
            'experimental_toughness_vs_terminal_reward',
            'experimental_strength_vs_terminal_reward',
        ])
    ].to_dict('records'),
    'primary_topk_10_percent': topk[(topk['aggregation'] == 'type_balanced') & (topk['top_fraction'] == 0.1)].to_dict('records'),
    'tables': {path.name: str(path) for path in sorted(TABLE_DIR.glob('*.csv'))},
    'figures': [str(path) for path in sorted(FIG_DIR.glob('*.png'))],
}
with open(TABLE_DIR / 'spider_silkom_summary.json', 'w') as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2)[:5000])

### Interpretation

The current structure-based predictor is not a direct spider silk material-property predictor. In the type-balanced taxon-level aggregation, global correlations between predicted SMD-like structure scores and material `toughness` / `tensile_strength` are weak. Top-k enrichment is also mostly near or below random baseline, with only terminal reward showing modest top-10% enrichment for material toughness.

This is scientifically reasonable: material properties of silk are emergent outcomes of multiple proteins, spinning conditions, hierarchical assembly, fiber morphology, crystallinity, water content, diameter, and experimental protocol. A single-protein structural reward can still be useful as a **candidate protein quality prior**, but it should not be used alone as the final screening objective for material-level silk performance.

Recommended next step: build a multi-instance / taxon-level model that combines per-protein structure rewards with composition features (`silk_types_included`, amino-acid composition, poly-A, GPGXX motifs, total length, fragment count) and material/fiber metadata when available.